# TRIBE v2: Colab text-to-brain demo

Based on [Meta's TRIBE v2](https://github.com/facebookresearch/tribev2). License: CC BY-NC 4.0. Predictions are a research model's average cortical response to synthesized audio.

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run **Install**, then **Runtime → Restart session**. Run the remaining cells in order; do not rerun Install after restarting.
3. The default prediction mode is **audio_only**. It uses Wav2Vec-BERT and TRIBE, without downloading or authenticating to Llama. The text is synthesized with gTTS; WhisperX provides word timings.
4. To include the language channel, set `PREDICTION_MODE = "full"` in the prediction cell. That mode requires approved Hugging Face access to `meta-llama/Llama-3.2-3B` and authentication with `huggingface_hub.notebook_login()`. Never commit a token.

Install repairs NumPy and requires a restart to discard cached binaries. Torch and Torchaudio are kept at compatible versions. WhisperX uses its own Python 3.11 environment and a shared NLTK data directory.

The audio-only path and cortical visualization were executed successfully on Colab T4. This mode excludes Llama's language features; it should not be treated as equivalent to the full model. A complete fresh installation on the latest Colab runtime has not been verified.

## 1. Install

In [ ]:
# Install into the actual notebook interpreter, not an implicit uv environment.
import sys
import shutil
import subprocess

if sys.version_info < (3, 11):
    raise RuntimeError("TRIBE v2 requires Python 3.11 or newer.")
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.16"], check=True)
TRIBE_REVISION = "af58661791a351a448a489042a28f6c37e1c14b7"
subprocess.run([
    "uv", "pip", "install", "--python", sys.executable,
    "--reinstall-package", "numpy",
    f"tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git@{TRIBE_REVISION}",
    "numpy==2.2.6", "torchaudio==2.6.0", "transformers==5.13.1", "nltk>=3.9.1,<4",
], check=True)
# A fresh subprocess verifies the files on disk without using cached imports.
subprocess.run([
    sys.executable, "-c",
    "import numpy as np; import numpy._core.strings; "
    "assert np.__version__ == '2.2.6'; print('Fresh NumPy import: OK', np.__version__)",
], check=True)
_TRIBE_RESTART_REQUIRED = True
print("Installation complete. Runtime → Restart session, then start at section 2.")

## 2. Load model (after runtime restart)

In [ ]:
if globals().get("_TRIBE_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the session after Install, then run from section 2. Do not rerun Install.")
import sys
import importlib.metadata as metadata
import numpy as np
import numpy._core.strings
if np.__version__ != "2.2.6" or np.__version__ != metadata.version("numpy"):
    raise RuntimeError("NumPy versions differ. Run Install, restart the session, and retry.")
from tribev2.demo_utils import TribeModel, download_file
from pathlib import Path
import torch
import torchaudio
if torch.__version__.split('+')[0] != torchaudio.__version__.split('+')[0]:
    raise RuntimeError('Torch and Torchaudio versions differ. Run Install and restart the session.')

CACHE_FOLDER = Path("/content/cache").resolve()
CACHE_FOLDER.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)
print("Model loaded")

## 3. Prepare WhisperX and NLTK data

WhisperX uses an isolated Python 3.11 environment; its Torch dependencies do not replace TRIBE's dependencies. The reproduced transcription failure was `LookupError: Resource punkt_tab not found`. A shared `NLTK_DATA` directory makes the downloaded resource visible inside the WhisperX subprocess. Failed runs retain stdout/stderr in `cache/whisperx_last.log`.

The GPU path uses float16, while CPU uses int8. Word alignment remains enabled; words with missing or invalid timestamps are excluded rather than assigned invented timings.

In [ ]:
import tribev2.eventstransforms as _evt
import pandas as pd
import nltk
import os
import json
import math
import subprocess
import tempfile

WHISPERX_VERSION = "3.8.6"
NLTK_DIR = str((CACHE_FOLDER / "nltk_data").resolve())
os.environ["NLTK_DATA"] = NLTK_DIR
if NLTK_DIR not in nltk.data.path:
    nltk.data.path.insert(0, NLTK_DIR)
for resource in ("punkt", "punkt_tab"):
    if not nltk.download(resource, download_dir=NLTK_DIR, raise_on_error=True):
        raise RuntimeError(f"NLTK download failed: {resource}")
nltk.data.find("tokenizers/punkt_tab/english/")

def _patched_transcript(wav_filename: Path, language: str) -> pd.DataFrame:
    codes = {"english": "en", "french": "fr", "spanish": "es", "dutch": "nl", "chinese": "zh"}
    if language not in codes:
        raise ValueError(f"Unsupported language: {language}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}
    env["NLTK_DATA"] = NLTK_DIR
    with tempfile.TemporaryDirectory() as output_dir:
        cmd = ["uvx", "--python", "3.11", f"whisperx=={WHISPERX_VERSION}", str(wav_filename),
               "--model", "large-v3", "--language", codes[language],
               "--device", device, "--compute_type", compute_type,
               "--batch_size", "4", "--output_dir", output_dir, "--output_format", "json"]
        if language == "english":
            cmd += ["--align_model", "WAV2VEC2_ASR_LARGE_LV60K_960H"]
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
        log_path = CACHE_FOLDER / "whisperx_last.log"
        log_path.write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
        if result.returncode:
            tail = "\n".join((result.stdout + "\n" + result.stderr).splitlines()[-45:])
            raise RuntimeError(f"WhisperX exited with {result.returncode}. Full log: {log_path}\n{tail}")
        json_path = Path(output_dir) / f"{Path(wav_filename).stem}.json"
        if not json_path.is_file():
            raise RuntimeError(f"WhisperX produced no JSON transcript. See {log_path}")
        transcript = json.loads(json_path.read_text(encoding="utf-8"))
    words = []
    for i, segment in enumerate(transcript.get("segments", [])):
        sentence = segment.get("text", "").replace('"', "")
        for word in segment.get("words", []):
            if "start" not in word or "end" not in word:
                continue
            start, end = word["start"], word["end"]
            if not isinstance(start, (int, float)) or not isinstance(end, (int, float)):
                continue
            if not math.isfinite(start) or not math.isfinite(end) or start < 0 or end <= start:
                continue
            words.append({"text": word["word"].replace('"', ""), "start": start,
                          "duration": end - start, "sequence_id": i, "sentence": sentence})
    if not words:
        raise RuntimeError(f"No aligned words found. See {log_path}")
    return pd.DataFrame(words, columns=["text", "start", "duration", "sequence_id", "sentence"])

_evt.ExtractWordsFromAudio._get_transcript_from_audio = staticmethod(_patched_transcript)
print("WhisperX workaround ready; shared NLTK resources:", NLTK_DIR)

## 4. Test text

Replace the English example with your own text after the first successful run. Text is converted to speech, so the output describes predicted response to **hearing**, not silently reading, a post. The original demo supports only certain languages.

In [ ]:
text = """To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune."""
text_path = CACHE_FOLDER / "sample.txt"
text_path.write_text(text, encoding="utf-8")
df = model.get_events_dataframe(text_path=text_path)
assert "Word" in df.type.values, "Transcription returned no Word events."
df.to_csv(CACHE_FOLDER / "text_events.csv", index=False)
display(df.head(8))
print("Events:", df.type.value_counts().to_dict())

In [ ]:
# "audio_only" does not load Llama. "full" requires approved Llama access.
PREDICTION_MODE = "audio_only"

import time
import json

def _predict_with_channels(model, events, channels):
    previous_features = list(model.data.features_to_use)
    previous_workers = model.data.num_workers
    previous_batch_size = model.data.batch_size
    try:
        model.data.features_to_use = list(channels)
        model.data.num_workers = 0
        model.data.batch_size = 1
        print("Active feature channels:", model.data.features_to_use)
        return model.predict(events=events)
    finally:
        model.data.features_to_use = previous_features
        model.data.num_workers = previous_workers
        model.data.batch_size = previous_batch_size

if PREDICTION_MODE == "audio_only":
    prediction_events = df.loc[df.type == "Audio"].copy().reset_index(drop=True)
    assert len(prediction_events) > 0
    assert set(prediction_events.type) == {"Audio"}
    prediction_channels = ["audio"]
elif PREDICTION_MODE == "full":
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import GatedRepoError
    try:
        hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
    except GatedRepoError:
        raise RuntimeError(
            "Full prediction requires approved Llama-3.2-3B access. "
            "Request it at https://huggingface.co/meta-llama/Llama-3.2-3B, "
            "then run: from huggingface_hub import notebook_login; notebook_login(). "
            "Alternatively use PREDICTION_MODE = 'audio_only'."
        ) from None
    prediction_events = df
    prediction_channels = ["text", "audio", "video"]
else:
    raise ValueError("PREDICTION_MODE must be 'audio_only' or 'full'.")

started = time.perf_counter()
preds, segments = _predict_with_channels(model, prediction_events, prediction_channels)
assert preds.ndim == 2 and preds.shape[0] > 0 and preds.shape[1] > 0
assert len(segments) == preds.shape[0] and np.isfinite(preds).all()
prediction_path = CACHE_FOLDER / f"{PREDICTION_MODE}_brain_activity.npy"
np.save(prediction_path, preds)
prediction_summary = {
    "mode": PREDICTION_MODE,
    "shape": list(preds.shape),
    "finite": bool(np.isfinite(preds).all()),
    "minimum": float(preds.min()),
    "maximum": float(preds.max()),
    "mean": float(preds.mean()),
    "std": float(preds.std()),
    "elapsed_seconds": round(time.perf_counter() - started, 3),
    "audio_duration_seconds": float(df.loc[df.type == "Audio", "duration"].sum()),
}
(CACHE_FOLDER / f"{PREDICTION_MODE}_summary.json").write_text(
    json.dumps(prediction_summary, indent=2), encoding="utf-8")
print(json.dumps(prediction_summary, indent=2))
print("Saved:", prediction_path)

In [ ]:
from tribev2.plotting import PlotBrain
import matplotlib.pyplot as plt

selected_steps = [i for i in [0, 3, 6, 9] if i < len(preds)]
bound = max(float(np.percentile(np.abs(preds), 99)), 1e-6)
plotter = PlotBrain(mesh="fsaverage5")
fig = plotter.plot_timesteps(
    {f"TRIBE: {PREDICTION_MODE}": preds[selected_steps]},
    timestamps=[float(i * model.data.TR) for i in selected_steps],
    show_stimuli=False, cmap="RdBu_r", vmin=-bound, vmax=bound,
)
fig.suptitle(f"TRIBE v2 — {PREDICTION_MODE} predicted cortical activity", y=1.08)
brain_map_path = CACHE_FOLDER / f"{PREDICTION_MODE}_brain_map.png"
fig.savefig(brain_map_path, dpi=180, bbox_inches="tight")
print("Saved:", brain_map_path)
plt.show()

## 5. Optional video example

In [ ]:
video_path = CACHE_FOLDER / "sample_video.mp4"
download_file("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", video_path)
df_video = model.get_events_dataframe(video_path=video_path)
display(df_video.head(8))

In [ ]:
video_preds, video_segments = model.predict(events=df_video)
print("Video prediction shape:", video_preds.shape)

## Verification — 12 October 2026

**Audio-only prediction passed on Colab T4**, using the synthesized Shakespeare example (9.648 seconds). Wav2Vec-BERT and TRIBE produced **10 × 20,484 float32 values**, all finite, in 16.37 seconds including the audio model download. A cortical map at t=0,3,6,9 seconds was rendered and saved. Llama was not loaded.

The first audio attempt failed because Torchaudio 2.11.0+cu128 was paired with Torch 2.6.0+cu124 (`undefined symbol: aoti_torch_abi_version`). Installing Torchaudio 2.6.0 restored compatibility. The notebook now pins this version during Install.

The WhisperX `punkt_tab` failure was reproduced and fixed by downloading `punkt` and `punkt_tab` and sharing `NLTK_DATA`. WhisperX 3.8.6 directly returned 25 aligned words. The Colab kernel used Python 3.12.13, NumPy 2.2.6 and Torch 2.6.0.

A separate clean NumPy 2.2.6 string import passed on Windows Python 3.13.13. The old NumPy AttributeError was not reproduced in the active Colab session; reinstall and restart address the suspected mixed-library state. A complete fresh installation on the latest Colab runtime remains unverified.

**Full text+audio prediction remains blocked** by `401 Unauthorized` / `GatedRepoError` for Llama-3.2-3B. Audio-only output is a reduced-input experiment, not a validation of the full model's accuracy.